# AquaInsight — Task 8: KNN and MICE Imputation

**Internship Project:** Predicting Water Quality Index for Comprehensive Water Assessment

This notebook is one of the nine independent GitHub deliverables. It can be run separately using the supplied water-quality CSV.

## Step-by-step approach

1. Load the supplied dataset.
2. Perform the task-specific analysis.
3. Display quantitative results.
4. Interpret the results for downstream water-quality modeling.
5. Preserve data for review rather than making unsupported automatic corrections.

In [ ]:
# Common setup — AquaInsight Water Quality Internship

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

try:
    from rapidfuzz.fuzz import ratio, token_set_ratio
except ImportError:
    raise ImportError("Install RapidFuzz first: pip install rapidfuzz")

possible_paths = [
    Path("Water Quality(1).csv"),
    Path("Water Quality.csv"),
    Path("../data/Water Quality(1).csv"),
    Path("../data/Water Quality.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the supplied CSV beside this notebook.")

df = pd.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head())


In [ ]:
# Build a sample-level water-quality matrix for the task
selected = [
    "pH",
    "Turbidity",
    "Specific conductance",
    "Nitrate",
    "Total Phosphorus, mixed forms",
    "Dissolved oxygen (DO)"
]

wide = (
    df[df["CharacteristicName"].isin(selected)]
    .assign(
        sample_key=lambda x:
            x["MonitoringLocationID"].astype(str) + "_" +
            x["ActivityStartDate"].astype(str)
    )
    .pivot_table(
        index="sample_key",
        columns="CharacteristicName",
        values="ResultValue",
        aggfunc="median"
    )
)

display(wide.head())
print("Wide matrix shape:", wide.shape)


# Task 8 — KNN and MICE Imputation Comparison

### Internship requirement
Implement and compare **K-Nearest Neighbors (KNN)** and **Multiple Imputation by Chained Equations (MICE/Iterative Imputation)** using validation against artificially masked observed values.

### Steps
1. Select a compact numeric feature matrix.
2. Randomly mask a known subset of observed values.
3. Apply KNN imputation.
4. Apply Iterative/MICE-style imputation.
5. Compare predictions with the original known values.
6. Use RMSE as the validation metric.
7. Select the method based on validation performance and data characteristics.


In [17]:
def imputation_validation(data, missing_rate=0.20, random_state=42):
    observed = data.select_dtypes(include="number").copy()
    observed = observed.dropna(axis=1, how="all")

    rng = np.random.default_rng(random_state)
    mask = rng.random(observed.shape) < missing_rate
    mask &= observed.notna().to_numpy()

    masked = observed.mask(mask)
    results = []

    imputers = {
        "Median": masked.fillna(masked.median()),
        "KNN": pd.DataFrame(
            KNNImputer(n_neighbors=5).fit_transform(masked),
            index=masked.index,
            columns=masked.columns
        ),
        "Iterative": pd.DataFrame(
            IterativeImputer(
                random_state=random_state,
                max_iter=10
            ).fit_transform(masked),
            index=masked.index,
            columns=masked.columns
        )
    }

    for method, imputed in imputers.items():
        actual = observed.to_numpy()[mask]
        predicted = imputed.to_numpy()[mask]
        valid = np.isfinite(actual) & np.isfinite(predicted)

        results.append({
            "method": method,
            "rmse": np.sqrt(
                mean_squared_error(actual[valid], predicted[valid])
            ),
            "masked_values_evaluated": int(valid.sum())
        })

    return pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)


imputation_input = wide.dropna(axis=1, how="all").copy()

# Keep the demonstration computationally manageable.
imputation_sample = imputation_input.sample(
    min(5000, len(imputation_input)), random_state=42
)

imputation_results = imputation_validation(imputation_sample)
display(imputation_results.round(4))

print("Lower RMSE indicates better recovery of artificially masked observed values in this validation experiment.")


,method,rmse,masked_values_evaluated
0,Iterative,58.2519,3944
1,KNN,67.5613,3944
2,Median,72.7317,3944


Lower RMSE indicates better recovery of artificially masked observed values in this validation experiment.


## Task 8 — Conclusion

The analysis above completes the requested **KNN and MICE Imputation** component of the AquaInsight internship assignment. Results should be interpreted together with domain requirements and the official WQI definition when it becomes available.